# INTERVAL: API download and split exports

Preserve all Places, separating those with multiple Submissions from those with zero or one.
CSV exports contain both groups; GeoJSON and Shapefile contain only the zero-or-one group.
The legacy output label `valid_submissions` describes this group, not scientific validation.

See [the individual guide](../docs/INTERVAL_process_data_from_API_and_generated_files.md).
Install pandas, requests and fiona in your selected Python kernel before running.


In [ ]:
from pathlib import Path
import sys
import json

HERE = Path.cwd().resolve()
ROOT = HERE.parent if HERE.name == 'src' else HERE
if not (ROOT / 'src' / 'project_paths.py').is_file():
    raise RuntimeError('Start Jupyter from the repository root or src/')
sys.path.insert(0, str(ROOT / 'src'))
from project_paths import project_root
from INTERVAL_process_data_from_API_and_generated_files import run_export
from INTERVAL_process_data import read_csv
ROOT = project_root()


## 1. Configure

Set your dataset ID. The example ID is 306; replace it for your export.
Leave `SOURCE_ZIP = None` for a live API download, or supply a local archive Path.
Each run gets its own folder. Set `SKIP_SHAPEFILE = True` for CSV/GeoJSON only.


In [ ]:
DATASET_ID = 306
SOURCE_ZIP = None
OUTPUT_DIR = ROOT / 'outputs' / 'interval_api_split'
SKIP_SHAPEFILE = False


## 2. Download, validate and export

This cell runs the complete shared pipeline and writes the outputs. Missing or duplicate
IDs stop the run. Invalid coordinates are kept as null geometries and reported.
If Shapefile text exceeds 254 UTF-8 bytes, use CSV/GeoJSON only rather than truncate it.
Restart the kernel after editing the Python module.


In [ ]:
run_dir = run_export(DATASET_ID, output_dir=OUTPUT_DIR, source_zip=SOURCE_ZIP,
                     skip_shapefile=SKIP_SHAPEFILE)
report = json.loads((run_dir / 'processing_report.json').read_text())
print(f'Completed export: {run_dir}')
report


## 3. Inspect source and processed data

Places without Submissions remain in the zero-or-one output. Unmatched Submissions
are excluded from processed outputs but remain in the raw CSV and are counted in the report.


In [ ]:
places_df = read_csv(run_dir / 'csv_files' / 'places.csv')
submissions_df = read_csv(run_dir / 'csv_files' / 'submissions.csv')
places_df.head()


In [ ]:
submissions_df.head()


In [ ]:
zero_or_one_df = read_csv(run_dir / 'csv_files' / f'data_valid_submissions_dataset_id_{DATASET_ID}.csv')
multiple_df = read_csv(run_dir / 'csv_files' / f'data_multiple_submissions_dataset_id_{DATASET_ID}.csv')
zero_or_one_df.head()


In [ ]:
multiple_df.head()


## 4. Locate the ZIP

The ZIP includes the original CSV pair, processed CSVs, GeoJSON, report and all five
Shapefile components when enabled. No attachment images are downloaded.


In [ ]:
zip_path = run_dir / f'dataset_id_{DATASET_ID}_outputs.zip'
print(zip_path)
